# Phase 3 — Numerical Analysis with NumPy


In [1]:
# === Setup ===
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

## Load data

In [2]:
# --- Raw data ---
con = sqlite3.connect(RAW / "ecommerce.db")
orders = pd.read_sql("SELECT * FROM orders", con)
con.close()
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders_completed = orders[orders["status"] == "completed"].copy()

catalog = pd.read_csv(RAW / "product_catalog_2024.csv")
catalog = catalog.rename(columns={"SKU": "product_id", "in_stock_units": "stock_units"})

# --- Phase 2 outputs ---
order_items_clean = pd.read_csv(PROCESSED / "clean_order_items.csv")
category_month_revenue = pd.read_csv(PROCESSED / "category_month_revenue.csv", index_col=0)

print("orders:", orders.shape)
print("orders_completed:", orders_completed.shape)
print("order_items_clean:", order_items_clean.shape)
print("category_month_revenue:", category_month_revenue.shape)

orders: (9000, 5)
orders_completed: (5101, 5)
order_items_clean: (20176, 8)
category_month_revenue: (6, 36)


---
## Part 1 — RFM Segmentation
**Goal:** score every customer on Recency (days since last order), Frequency (number of completed orders), and Monetary value (total spend), then combine into one segmentation score.

In [3]:
# Join order_items -> completed orders to get customer + date on every purchased item
order_revenue = order_items_clean.merge(
    orders_completed[["order_id", "customer_id", "order_date"]], on="order_id"
)

reference_date = order_revenue["order_date"].max() + pd.Timedelta(days=1)

rfm_raw = order_revenue.groupby("customer_id").agg(
    last_order_date=("order_date", "max"),
    frequency=("order_id", "nunique"),
    monetary=("net_revenue", "sum"),
).reset_index()

rfm_raw["recency"] = (reference_date - rfm_raw["last_order_date"]).dt.days

recency = rfm_raw["recency"].to_numpy()
frequency = rfm_raw["frequency"].to_numpy()
monetary = rfm_raw["monetary"].to_numpy()

print(f"Customers scored: {len(recency)}")
print("preview (first 5):", recency[:5], frequency[:5], monetary[:5])

Customers scored: 2191
preview (first 5): [373  80 280  73 154] [2 2 6 1 1] [ 1731.762   2567.687  12657.7395  2723.514   1670.3825]


In [4]:
# Manual quintile scoring (replaces pandas.qcut) using np.percentile + np.digitize
def score_quintile(values: np.ndarray, reverse: bool = False) -> np.ndarray:
    """Splits values into 5 buckets by percentile, returns a 1(worst)-5(best) score.
    reverse=True flips direction (used for recency, where LOWER is better)."""
    cutoffs = np.percentile(values, [20, 40, 60, 80])
    scores = np.digitize(values, cutoffs) + 1
    if reverse:
        scores = 6 - scores
    return scores

rfm_raw["r_score"] = score_quintile(recency, reverse=True)
rfm_raw["f_score"] = score_quintile(frequency, reverse=False)
rfm_raw["m_score"] = score_quintile(monetary, reverse=False)

# Combined segmentation score: equal-weighted sum, range 3-15
rfm_raw["rfm_score"] = rfm_raw["r_score"] + rfm_raw["f_score"] + rfm_raw["m_score"]

def label_segment(score: int) -> str:
    if score >= 13: return "Champions"
    elif score >= 10: return "Loyal"
    elif score >= 7: return "At Risk"
    else: return "Lost"

rfm_raw["segment"] = rfm_raw["rfm_score"].apply(label_segment)
rfm_raw["segment"].value_counts()

segment
Loyal        710
At Risk      567
Champions    504
Lost         410
Name: count, dtype: int64

In [5]:
# Sanity check ONLY — compare manual monetary scoring against pandas.qcut
check = pd.qcut(rfm_raw["monetary"], 5, labels=False, duplicates="drop") + 1
agreement = (check == rfm_raw["m_score"]).mean()
print(f"Manual scoring agrees with pd.qcut on {agreement:.1%} of customers")

Manual scoring agrees with pd.qcut on 99.8% of customers


---
## Part 2 — Similarity / Recommendation
**Goal:** find products that tend to be bought together (cosine similarity), then recommend 3 unbought products to 5 sample customers.

In [6]:
# Customer x product purchase matrix (rows = customers, columns = products)
purchases = order_items_clean.merge(orders_completed[["order_id", "customer_id"]], on="order_id")
purchases = purchases[purchases["quantity"] > 0]  # exclude returns

purchase_matrix = purchases.pivot_table(
    index="customer_id", columns="product_id", values="quantity", aggfunc="sum", fill_value=0
)
purchase_array = purchase_matrix.to_numpy()
print("purchase matrix shape:", purchase_array.shape)

purchase matrix shape: (2180, 300)


In [7]:
# Cosine similarity between PRODUCTS, computed by hand:
# cos_sim(A, B) = (A . B) / (||A|| * ||B||)
product_vectors = purchase_array.T  # (num_products, num_customers)

dot_products = product_vectors @ product_vectors.T
norms = np.linalg.norm(product_vectors, axis=1)
norm_matrix = np.outer(norms, norms)
norm_matrix[norm_matrix == 0] = 1e-9

product_similarity = dot_products / norm_matrix
print("similarity matrix shape:", product_similarity.shape)

similarity matrix shape: (300, 300)


In [8]:
product_ids = purchase_matrix.columns.to_numpy()
customer_ids = purchase_matrix.index.to_numpy()

def recommend_for_customer(customer_id, n=3):
    """Scores every product by similarity to what this customer already bought,
    excludes products they already own, returns the top n."""
    cust_row = purchase_array[customer_ids == customer_id][0]
    already_bought = set(product_ids[cust_row > 0])
    scores = product_similarity @ cust_row
    ranked = np.argsort(-scores)
    recs = []
    for idx in ranked:
        pid = product_ids[idx]
        if pid not in already_bought:
            recs.append(pid)
        if len(recs) == n:
            break
    return recs

sample_customers = customer_ids[:5]
recommendations = {}
for cid in sample_customers:
    recs = recommend_for_customer(cid)
    recommendations[cid] = recs
    print(f"Customer {cid} -> recommended products: {recs}")

Customer 1 -> recommended products: [np.int64(168), np.int64(74), np.int64(233)]
Customer 2 -> recommended products: [np.int64(198), np.int64(148), np.int64(137)]
Customer 3 -> recommended products: [np.int64(53), np.int64(70), np.int64(68)]
Customer 5 -> recommended products: [np.int64(109), np.int64(132), np.int64(158)]
Customer 7 -> recommended products: [np.int64(82), np.int64(103), np.int64(66)]


In [9]:
# Sanity check ONLY — compare against sklearn on a small slice
from sklearn.metrics.pairwise import cosine_similarity
sklearn_check = cosine_similarity(product_vectors[:20])
manual_check = product_similarity[:20, :20]
print(f"Max difference vs sklearn: {np.abs(sklearn_check - manual_check).max():.10f}")

Max difference vs sklearn: 0.0000000000


---
## Part 3 — Regression via Normal Equation
**Goal:** fit a trend line through monthly revenue using β = (XᵀX)⁻¹Xᵀy directly, not a fitted library model.

In [10]:
monthly_revenue = category_month_revenue.sum(axis=0).sort_index()
print(monthly_revenue)

n_months = len(monthly_revenue)
X_raw = np.arange(n_months).reshape(-1, 1)
y = monthly_revenue.to_numpy().reshape(-1, 1)

X = np.hstack([np.ones((n_months, 1)), X_raw])  # add intercept column

# === Normal Equation ===
XtX = X.T @ X
XtX_inv = np.linalg.inv(XtX)
beta = XtX_inv @ X.T @ y

intercept, slope = beta[0, 0], beta[1, 0]
print(f"Model: revenue = {intercept:.2f} + {slope:.2f} * month_index")

2022-01    290409.5380
2022-02    160997.5940
2022-03    223735.3185
2022-04    169965.4915
2022-05    284261.3355
2022-06    192353.0990
2022-07    347325.8425
2022-08    212186.0455
2022-09    243855.2580
2022-10    299473.7065
2022-11    262704.6065
2022-12    276292.1780
2023-01    248306.7020
2023-02    264517.0190
2023-03    298466.1410
2023-04    230378.9830
2023-05    451297.3615
2023-06    349487.3460
2023-07    360291.3015
2023-08    278069.1915
2023-09    445457.7010
2023-10    327280.8605
2023-11    435339.7490
2023-12    432412.3685
2024-01    453426.7830
2024-02    400268.8805
2024-03    439162.7320
2024-04    435938.0480
2024-05    518127.9870
2024-06    498260.1270
2024-07    520034.8300
2024-08    594403.5440
2024-09    481085.0850
2024-10    448012.5810
2024-11    610885.7660
2024-12    484326.2315
dtype: float64
Model: revenue = 182785.15 + 10140.53 * month_index


In [11]:
# R^2 computed manually
y_pred = X @ beta
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r_squared = 1 - (ss_res / ss_tot)
print(f"R² = {r_squared:.4f}")

# Forecast next 2 months
future_idx = np.array([[1, n_months], [1, n_months + 1]])
future_revenue = future_idx @ beta
print(f"Forecast next 2 months: {future_revenue.flatten()}")

R² = 0.7786
Forecast next 2 months: [547844.11538254 557984.64214045]


In [12]:
# Sanity check ONLY — compare against sklearn
from sklearn.linear_model import LinearRegression
sklearn_model = LinearRegression().fit(X_raw, y)
print(f"sklearn intercept/slope: {sklearn_model.intercept_[0]:.2f}, {sklearn_model.coef_[0][0]:.2f}")
print(f"Manual  intercept/slope: {intercept:.2f}, {slope:.2f}")

sklearn intercept/slope: 182785.15, 10140.53
Manual  intercept/slope: 182785.15, 10140.53


---
## Part 4 — Monte Carlo Simulation
**Goal:** simulate demand thousands of times for 3 products, estimate the probability current stock runs out, with a 95% confidence interval.

In [13]:
demand_history = purchases.merge(orders_completed[["order_id", "order_date"]], on="order_id")
demand_history["month"] = demand_history["order_date"].dt.to_period("M").astype(str)

monthly_demand = demand_history.groupby(["product_id", "month"])["quantity"].sum().reset_index()
demand_stats = monthly_demand.groupby("product_id")["quantity"].agg(["mean", "std"]).reset_index()
demand_stats = demand_stats.dropna()

chosen_products = demand_stats.sort_values("mean", ascending=False).head(3)
chosen_products = chosen_products.merge(catalog[["product_id", "stock_units"]], on="product_id", how="left")
chosen_products = chosen_products.dropna(subset=["stock_units"])
print(chosen_products)

   product_id      mean       std  stock_units
0          72  3.181818  1.680033          142
1         293  3.047619  2.290768          305
2         274  2.888889  1.717183           27


In [14]:
np.random.seed(42)
n_trials = 5000
results = []

for _, row in chosen_products.iterrows():
    pid, mean_d, std_d, stock = row["product_id"], row["mean"], row["std"], row["stock_units"]

    simulated_demand = np.random.normal(loc=mean_d, scale=std_d, size=n_trials)
    simulated_demand = np.clip(simulated_demand, 0, None)

    stockouts = simulated_demand > stock
    stockout_prob = stockouts.mean()

    se = np.sqrt(stockout_prob * (1 - stockout_prob) / n_trials)
    ci_lower = max(0, stockout_prob - 1.96 * se)
    ci_upper = min(1, stockout_prob + 1.96 * se)

    results.append({
        "product_id": pid, "stock": stock, "mean_monthly_demand": mean_d,
        "stockout_probability": round(stockout_prob, 4),
        "ci_95_lower": round(ci_lower, 4), "ci_95_upper": round(ci_upper, 4),
    })

monte_carlo_results = pd.DataFrame(results)
monte_carlo_results

,product_id,stock,mean_monthly_demand,stockout_probability,ci_95_lower,ci_95_upper
0,72.0,142.0,3.181818,0.0,0,0.0
1,293.0,305.0,3.047619,0.0,0,0.0
2,274.0,27.0,2.888889,0.0,0,0.0


In [15]:
# Sanity check ONLY — trivial case: stock far below demand should give ~1.0 probability
test_demand = np.random.normal(loc=100, scale=10, size=5000)
test_stockout_rate = (test_demand > 10).mean()
print(f"Sanity check (stock far below demand): {test_stockout_rate:.4f}")

Sanity check (stock far below demand): 1.0000


---
## Save Phase 3 outputs
So Phase 4's notebook can reload these the same self-contained way.

In [16]:
rfm_raw.to_csv(PROCESSED / "rfm_segments.csv", index=False)
monte_carlo_results.to_csv(PROCESSED / "monte_carlo_stockout.csv", index=False)
np.save(PROCESSED / "product_similarity_matrix.npy", product_similarity)
np.save(PROCESSED / "regression_beta.npy", beta)

print("Phase 3 outputs saved to data/processed/")

Phase 3 outputs saved to data/processed/
